# ESMFold Structure Prediction

![ESMFold Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/esmfold/hero.png)

This notebook demonstrates protein structure prediction using ESMFold. We cover a single-sequence prediction with result inspection and visualization, followed by a batch prediction over multiple sequences. ESMFold runs without multiple sequence alignments, making it fast enough for iterative design workflows where hundreds of candidate sequences need structural evaluation.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esmfold")
display_overview("esmfold")
display_docs_section("esmfold", "Background")

# ESMFold

First released in 2022, ESMFold is Meta AI's protein language model based structure predictor. It folds a protein from its sequence using the embeddings of ESM-2 in place of a multiple-sequence alignment, making it roughly an order of magnitude faster than alignment-based methods like AlphaFold2 at competitive but generally somewhat lower accuracy. This toolkit provides two tools: structure prediction for proteins and complexes from sequence, and a differentiable confidence pass that returns the gradient of an ESMFold confidence loss for sequence design.

ESMFold ([Lin et al., 2023](https://doi.org/10.1126/science.ade2574)) predicts a protein's 3D structure directly from its amino-acid sequence, without the multiple-sequence alignment (MSA) that AlphaFold2 depends on. AlphaFold2 infers which residues are in contact by reading coevolution across an alignment of homologous sequences. ESMFold instead relies on the ESM-2 protein language model, which has already internalized those evolutionary patterns by pre-training on hundreds of millions of natural sequences, so it works from the lone sequence with no alignment built at inference time. Skipping the alignment search makes ESMFold roughly an order of magnitude faster than AlphaFold2, at some cost in accuracy on targets where a deep, diverse MSA would otherwise help.

The sequence first runs through a frozen ESM-2 transformer (the released model uses the 3-billion-parameter ESM-2), which produces a per-residue representation. A folding trunk, a simplified stand-in for AlphaFold2's Evoformer, refines that representation, and a structure module reused essentially unchanged from AlphaFold2 then places each residue as a rigid backbone frame to produce all-atom coordinates. The whole prediction is recycled through these stages several times. Alongside the coordinates, ESMFold reports calibrated confidence: a per-residue predicted local distance difference test (pLDDT) for local reliability, a predicted aligned error (PAE) for the expected error in one residue's position when the structure is aligned on another, and a predicted template-modeling score (pTM) for overall fold confidence.

Meta AI open-sources the reference implementation at [facebookresearch/esm](https://github.com/facebookresearch/esm) under the MIT license; the released model is the `esmfold_v1` checkpoint, whose structure module is taken from the OpenFold reimplementation of AlphaFold2. Because the language model carries the structural signal, ESM-2's perplexity on a sequence correlates with how accurate the predicted structure will be, and accuracy continues to improve as the ESM-2 backbone is scaled up. ESMFold's speed made its headline application possible: Meta AI folded over 600 million metagenomic protein sequences and released them as the [ESM Metagenomic Atlas](https://esmatlas.com).

## Available tools

In [2]:
display_available_tools("esmfold")

- **`run_esmfold_gradient()`** — Differentiable ESMFold confidence loss and gradient w.r.t. target-chain logits
- **`run_esmfold()`** — Protein structure prediction using ESMFold

### `run_esmfold`

ESMFold predicts protein 3D structures directly from amino acid sequence without requiring multiple sequence alignments. It uses a large protein language model backbone to infer residue-level geometry, producing atomic coordinates along with per-residue pLDDT confidence scores and a global pTM score. The lack of an MSA step makes it substantially faster than alignment-dependent methods, which is especially valuable for screening large libraries of designed sequences.

In [3]:
from proto_tools import ESMFoldInput, ESMFoldConfig, run_esmfold

In [4]:
# Display input API reference
display_api_reference("esmfold", "input", "run_esmfold")

# Ubiquitin (76 residues): a small, well-ordered domain ESMFold folds with high confidence
ubiquitin_sequence = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"

# Create input
inputs = ESMFoldInput(complexes=[ubiquitin_sequence])

**Input** — `ESMFoldInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>complexes</code> | <code>list[Complex]</code> | required | List of complexes to predict structure for containing chains and entity types. |
| <code>msas</code> | <code>list[ComplexMSAs] &#124; None</code> | <code>None</code> | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
# Display config API reference
display_api_reference("esmfold", "config", "run_esmfold")

# Configure ESMFold
config = ESMFoldConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESMFoldConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on (e.g., 'cuda', 'cpu') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>include_pae_matrix</code> | <code>bool</code> | <code>False</code> | Attach the full per-residue PAE matrix. |
| <code>residue_idx_offset</code> | <code>int</code> | <code>512</code> | Residue numbering gap between chains in multi-chain structures |
| <code>chain_linker</code> | <code>str</code> | <code>'GGGGGGGGGGGGGGGGGGGGGGGGG'</code> | Sequence to link chains (default: 25 glycines) |
| <code>max_batch_residues</code> | <code>int</code> | <code>1200</code> | Starting cap on total residues per inference batch; auto-halved on CUDA OOM |
| <code>num_recycles</code> | <code>int</code> | <code>4</code> | Iterative refinement passes through ESMFold. Higher = more accurate but slower. |

In [6]:
# Run the prediction
result = run_esmfold(inputs, config)

Folding structures (ESMFold):   0%|          | 0/1 [00:00<?, ?structure/s]

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [7]:
# Display output API reference
display_api_reference("esmfold", "output", "run_esmfold")

ubiquitin_structure = result.structures[0]

# Print confidence metrics
print(f"Length:       {ubiquitin_structure.num_residues} residues")
print(f"Average pLDDT: {ubiquitin_structure.metrics.avg_plddt:.3f}")
print(f"pTM score:    {ubiquitin_structure.metrics.ptm:.3f}")

# Visualize predicted structure colored by pLDDT confidence
ubiquitin_structure.visualize()

**Output** — `ESMFoldOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[Structure]</code> | required | List of predicted structures, one per input complex. |

**Metrics** — `ESMFoldMetrics` (one set per `structures` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>avg_plddt</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>ptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>avg_pae</code> | <code>float</code> | <code>&gt;= 0</code> |  |  |
| <code>pae</code> | <code>list[list[float]]</code> | <code>&gt;= 0</code> |  |  |

Length:       76 residues
Average pLDDT: 0.858
pTM score:    0.829


### `run_esmfold_gradient`

The gradient tool runs ESMFold over a relaxed `(L, 20)` distribution for one or more designated target chains in a complex and returns the gradient of a weighted confidence loss (`pLDDT`, `pTM`, `pAE`) with respect to those input logits. The relaxed distribution is mixed into both ESM-2's word embeddings and ESMFold's own AA embedding, so the structure module's confidence heads can differentiate back to the input logits while the model parameters stay frozen. Use this as a structure-aware loss inside MCMC or gradient descent over relaxed protein sequences. Set `compute_gradient=False` for forward-only confidence scoring.

In [8]:
from proto_tools import (
    ESMFoldGradientConfig,
    ESMFoldGradientInput,
    run_esmfold_gradient,
)
from proto_tools.utils import one_hot_protein_logits


In [9]:
# Display input API reference
display_api_reference("esmfold", "input", "run_esmfold_gradient")

# Seed a relaxed target-chain distribution from a discrete sequence (sharpness=2.0
# yields a biased-but-not-saturated softmax target).
target_seq = "MKTAYIAKQR"
logits = one_hot_protein_logits(target_seq, sharpness=2.0)

inputs = ESMFoldGradientInput(
    logits=logits,
    chains=[target_seq],          # length must equal len(logits) for every target chain
    target_chain_indices=[0],     # which chains receive the relaxed distribution
)


**Input** — `ESMFoldGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>logits</code> | <code>list[list[float]]</code> | required | Relaxed sequence logits with shape (L, 20) in backend-specific amino-acid order. |
| <code>temperature</code> | <code>float</code> | <code>1.0</code> | Softmax temperature used to convert logits into a relaxed amino-acid probability distribution. |
| <code>chains</code> | <code>list[str]</code> | required | Complete protein-chain sequences for the ESMFold complex. |
| <code>target_chain_indices</code> | <code>list[int]</code> | <code>[0]</code> | Zero-based chain indices that receive the relaxed input logits. |

In [10]:
# Display config API reference
display_api_reference("esmfold", "config", "run_esmfold_gradient")

# Configure the differentiable confidence pass
config = ESMFoldGradientConfig(
    loss_weights={"plddt": 1.0, "ptm": 0.5},   # weighted sum of confidence terms
    soft=1.0,                                  # 1.0 = pure soft probabilities
    hard=0.0,                                  # set to 1.0 for Straight-Through Estimator
    compute_gradient=True,
    num_recycles=1,                            # keep recycles low for gradient passes
    device="cuda",                             # Change to "cpu" if no GPU available
)


**Config** — `ESMFoldGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on (e.g., 'cuda', 'cpu') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>include_pae_matrix</code> | <code>bool</code> | <code>False</code> | Attach the full per-residue PAE matrix. |
| <code>residue_idx_offset</code> | <code>int</code> | <code>512</code> | Residue numbering gap between chains in multi-chain structures |
| <code>chain_linker</code> | <code>str</code> | <code>'GGGGGGGGGGGGGGGGGGGGGGGGG'</code> | Sequence to link chains (default: 25 glycines) |
| <code>max_batch_residues</code> | <code>int</code> | <code>1200</code> | Starting cap on total residues per inference batch; auto-halved on CUDA OOM |
| <code>num_recycles</code> | <code>int</code> | <code>4</code> | Iterative refinement passes through ESMFold. Higher = more accurate but slower. |
| <code>loss_weights</code> | <code>dict[str, float]</code> | <code>{'plddt': 1.0}</code> | ESMFold confidence loss weights. Valid keys: plddt, ptm, pae. |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend hard argmax one-hot (0) to softmax probabilities (1). |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through hard-forward coefficient. |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring. |

In [11]:
# Run one differentiable confidence pass
gradient_result = run_esmfold_gradient(inputs, config)


Starting worker


Running esmfold


Loading ESMFold model: facebook/esmfold_v1 on cuda:0


Moving ESMFold to cuda:0 (fp16)


In [12]:
# Display output API reference
display_api_reference("esmfold", "output", "run_esmfold_gradient")

# Inspect the scalar loss, per-term metrics, and gradient shape
print(f"Weighted loss: {gradient_result.loss:.3f}")
print(f"Avg pLDDT:    {gradient_result.metrics['avg_plddt']:.3f}")
print(f"loss_plddt:   {gradient_result.metrics['loss_plddt']:.3f}")
if 'loss_ptm' in gradient_result.metrics:
    print(f"loss_ptm:     {gradient_result.metrics['loss_ptm']:.3f}")
assert gradient_result.gradient is not None  # backward pass enabled
print(f"Gradient shape: ({len(gradient_result.gradient)}, {len(gradient_result.gradient[0])})")

# The predicted structure is also returned, colored by pLDDT
gradient_result.structure.visualize()


**Output** — `ESMFoldGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gradient</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Gradient w.r.t. input logits. None when compute_gradient=False. |
| <code>loss</code> | <code>float</code> | required | Scalar objective for this sequence; lower is better (typically a positive negative log-likelihood) |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Auxiliary metrics reported alongside the scalar objective value. |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |
| <code>structure</code> | <code>Structure</code> | required | Predicted ESMFold complex structure. |

**Metrics** — `ESMFoldMetrics`

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>avg_plddt</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>ptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>avg_pae</code> | <code>float</code> | <code>&gt;= 0</code> |  |  |
| <code>pae</code> | <code>list[list[float]]</code> | <code>&gt;= 0</code> |  |  |

Weighted loss: 0.822
Avg pLDDT:    0.664
loss_plddt:   0.336
loss_ptm:     0.972
Gradient shape: (10, 20)


## Batch Predictions

Protein design workflows routinely generate dozens to hundreds of candidate sequences that each require structural evaluation. ESMFold supports batch input natively, processing all sequences in a single call with automatic GPU batching for memory safety. The output contains one predicted structure per input sequence in the same order as the input list.

In [13]:
# Four distinct, well-folded small proteins spanning different structural classes.
sequences = [
    "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",   # Protein G B1 domain (alpha/beta)
    "RPDFCLEPPYTGPCKARIIRYFYNAKAGLCQTFVYGGCRAKRNNFKSAEDCMRTCGGA",  # BPTI (Kunitz fold)
    "TTCCPSIVARSNFNVCRLPGTPEAICATYTGCIIIPGATCPGDYAN",            # Crambin (disulfide-rich)
    "MLSDEDFKAVFGMTRSAFANLPLWKQQNLKKEKGLF",                       # Villin headpiece HP36 (helical)
]

# Create batch input
batch_input = ESMFoldInput(complexes=sequences)

In [14]:
# Configure batch prediction
batch_config = ESMFoldConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [15]:
# Run the batch prediction
batch_result = run_esmfold(batch_input, batch_config)

Folding structures (ESMFold):   0%|          | 0/4 [00:00<?, ?structure/s]

In [16]:
# Inspect results for all predicted structures
for i, structure in enumerate(batch_result.structures):
    print(f"Structure {i + 1}:")
    print(f"  Length:        {structure.num_residues} residues")
    print(f"  Average pLDDT: {structure.metrics.avg_plddt:.3f}")
    print(f"  pTM score:     {structure.metrics.ptm:.3f}")

Structure 1:
  Length:        56 residues
  Average pLDDT: 0.852
  pTM score:     0.829
Structure 2:
  Length:        58 residues
  Average pLDDT: 0.908
  pTM score:     0.829
Structure 3:
  Length:        46 residues
  Average pLDDT: 0.499
  pTM score:     0.829
Structure 4:
  Length:        36 residues
  Average pLDDT: 0.870
  pTM score:     0.829


## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD, and as input to further computational workflows including molecular dynamics, docking, or inverse folding. The B-factor column in exported files contains the normalized pLDDT confidence scores (0.0-1.0), so coloring by B-factor in a viewer directly shows per-residue model confidence.

In [17]:
from pathlib import Path

# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export single-sequence prediction to PDB
result.export(name="ubiquitin_structure", export_path=output_dir, file_format="pdb")

# Export batch predictions to PDB
batch_result.export(name="batch_structures", export_path=output_dir, file_format="pdb")